# CityGuard — śmieci (YOLO12)

Fine-tuning `yolo12n.pt` na **pLitterStreet**: 13 064 zdjęcia z kamery na aucie, ~79 tys. boxów, pobocza w Azji (Tajlandia, Wietnam, Sri Lanka). To ten sam typ kadru co patrol — nie studio i nie plaża.

Źródło: [Zenodo 8288500](https://zenodo.org/records/8288500), kod i opis: [gicait/pLitter](https://github.com/gicait/pLitter), artykuł: [arXiv:2401.14719](https://arxiv.org/abs/2401.14719). Etykiety COCO → YOLO. Kosze (`trash bin` / podobne) wypadają — to nie jest śmieć. Resztę scalamy do `bag`, `bottle`, `pile`, `plastic`.

Dziury trenuj osobno w `ml/train_road.ipynb`.

Środowisko: **GPU T4**. `images.zip` ma 13,7 GB — darmowy Colab to uciągnie, byle nie trzymać jednocześnie pełnego RDD. Token Kaggle nie jest potrzebny.

In [ ]:
!pip -q install -U ultralytics

In [ ]:
import torch

!nvidia-smi
if not torch.cuda.is_available():
    raise SystemExit("Włącz GPU: Środowisko wykonawcze → Zmień typ środowiska → T4")
print("GPU:", torch.cuda.get_device_name(0))

## Konfiguracja

`MAX_IMAGES = 0` bierze cały pLitterStreet. Na krótką sesję ustaw `4000`. `SAVE_TO_DRIVE` jest opcjonalne — jak mount padnie, trening idzie dalej, wagi zostają w `/content`.

In [ ]:
MAX_IMAGES = 0  # 0 = cały zbiór
EPOCHS = 40
IMGSZ = 640
BATCH = 16
CONF = 0.35
SAVE_TO_DRIVE = True
ZENODO = "https://zenodo.org/records/8288500/files"

In [ ]:
import json
import random
import shutil
import subprocess
from collections import defaultdict
from pathlib import Path

import yaml
from PIL import Image
from ultralytics import YOLO

IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
RAW = Path("/content/raw/plitter")
DEST = Path("/content/datasets/litter")

# nazwa z COCO (po lower/_) → klasa CityGuard; None = pomiń (kosz, nie śmieć)
LITTER_MAP = {
    "bag": "bag",
    "plastic bag": "bag",
    "bottle": "bottle",
    "plastic bottle": "bottle",
    "cup": "plastic",
    "other plastic": "plastic",
    "plastic": "plastic",
    "sachet": "plastic",
    "straw": "plastic",
    "rope": "plastic",
    "face mask": "plastic",
    "facemask": "plastic",
    "mask": "plastic",
    "pile": "pile",
    "trash bin": None,
    "trashbin": None,
    "bin": None,
    "dustbin": None,
}
CLASS_NAMES = ["bag", "bottle", "pile", "plastic"]


def run(cmd: list[str]) -> None:
    print("+", " ".join(cmd))
    subprocess.check_call(cmd)


def download_zenodo(filename: str) -> Path:
    RAW.mkdir(parents=True, exist_ok=True)
    zip_path = RAW / filename
    url = f"{ZENODO}/{filename}?download=1"
    if not zip_path.exists() or zip_path.stat().st_size < 1_000_000:
        run(["wget", "-c", "-O", str(zip_path), url])
    return zip_path


def unzip_once(zip_path: Path, dest: Path) -> Path:
    marker = dest / ".extracted"
    if marker.exists():
        print("już rozpakowane:", dest.name)
        return dest
    dest.mkdir(parents=True, exist_ok=True)
    run(["unzip", "-q", "-o", str(zip_path), "-d", str(dest)])
    marker.write_text("ok", encoding="utf-8")
    return dest


def canon(name: str) -> str:
    return " ".join(name.lower().replace("-", " ").replace("_", " ").split())


def map_class(name: str) -> str | None:
    key = canon(name)
    if key in LITTER_MAP:
        return LITTER_MAP[key]
    for token, mapped in LITTER_MAP.items():
        if token in key:
            return mapped
    return "plastic"


def find_coco(root: Path) -> Path:
    jsons = sorted(root.rglob("*.json"))
    if not jsons:
        raise FileNotFoundError("pLitterStreet: brak pliku COCO (*.json)")
    for path in jsons:
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        if isinstance(data, dict) and "images" in data and "annotations" in data:
            return path
    raise FileNotFoundError(f"pLitterStreet: żaden JSON nie jest COCO, sprawdziłem {len(jsons)} plików")


def index_images(root: Path) -> dict[str, Path]:
    found: dict[str, Path] = {}
    for path in root.rglob("*"):
        if path.suffix.lower() in IMAGE_EXT:
            found.setdefault(path.name, path)
            found.setdefault(path.stem, path)
    return found


def coco_to_pairs(ann_root: Path, img_root: Path) -> list[tuple[Path, str]]:
    coco_path = find_coco(ann_root)
    print("COCO:", coco_path)
    coco = json.loads(coco_path.read_text(encoding="utf-8"))
    cats = {int(c["id"]): map_class(str(c.get("name", ""))) for c in coco.get("categories", [])}
    print("mapowanie klas COCO →", {c.get("name"): cats[int(c["id"])] for c in coco.get("categories", [])})
    files = index_images(img_root)
    by_id: dict[int, list[str]] = defaultdict(list)
    skipped_bin = 0
    for ann in coco.get("annotations", []):
        mapped = cats.get(int(ann["category_id"]))
        if mapped is None:
            skipped_bin += 1
            continue
        bbox = ann.get("bbox") or []
        if len(bbox) < 4:
            continue
        x, y, w, h = (float(v) for v in bbox[:4])
        if w <= 1 or h <= 1:
            continue
        by_id[int(ann["image_id"])].append((mapped, x, y, w, h))
    labeled: list[tuple[Path, str]] = []
    missing = 0
    for image in coco.get("images", []):
        boxes = by_id.get(int(image["id"]))
        if not boxes:
            continue
        name = image.get("file_name") or ""
        path = files.get(Path(name).name) or files.get(Path(name).stem)
        if path is None:
            missing += 1
            continue
        width = float(image.get("width") or 0)
        height = float(image.get("height") or 0)
        if width <= 1 or height <= 1:
            with Image.open(path) as handle:
                width, height = handle.size
        lines = []
        for mapped, x, y, w, h in boxes:
            xc = min(1.0, max(0.0, (x + w / 2) / width))
            yc = min(1.0, max(0.0, (y + h / 2) / height))
            nw = min(1.0, max(0.0, w / width))
            nh = min(1.0, max(0.0, h / height))
            if nw <= 0 or nh <= 0:
                continue
            lines.append(f"{CLASS_NAMES.index(mapped)} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")
        if lines:
            labeled.append((path, "\n".join(lines) + "\n"))
    print(f"obrazy z śmieciem: {len(labeled)}, pominięte kosze: {skipped_bin}, brak pliku: {missing}")
    if not labeled:
        raise FileNotFoundError("pLitterStreet: zero par po mapowaniu klas")
    random.Random(42).shuffle(labeled)
    if MAX_IMAGES and len(labeled) > MAX_IMAGES:
        labeled = labeled[:MAX_IMAGES]
        print("obcinam do MAX_IMAGES =", MAX_IMAGES)
    return labeled


def link_split(pairs: list[tuple[Path, str]], dest: Path, names: list[str]) -> Path:
    if dest.exists():
        shutil.rmtree(dest)
    for split in ("train", "val"):
        (dest / "images" / split).mkdir(parents=True)
        (dest / "labels" / split).mkdir(parents=True)
    val_count = max(1, int(len(pairs) * 0.1))
    if len(pairs) < 2:
        raise RuntimeError(f"Za mało obrazów: {len(pairs)}")
    for index, (image, label) in enumerate(pairs):
        split = "val" if index < val_count else "train"
        stem = f"{index:05d}_{image.stem}"
        target = dest / "images" / split / f"{stem}{image.suffix.lower()}"
        target.symlink_to(image.resolve())
        (dest / "labels" / split / f"{stem}.txt").write_text(label, encoding="utf-8")
    yaml_path = dest / "cityguard.yaml"
    yaml_path.write_text(
        yaml.safe_dump(
            {
                "train": str((dest / "images" / "train").resolve()),
                "val": str((dest / "images" / "val").resolve()),
                "names": {i: name for i, name in enumerate(names)},
            },
            sort_keys=False,
            allow_unicode=True,
        ),
        encoding="utf-8",
    )
    print(f"litter: {len(pairs) - val_count} train / {val_count} val, klasy {names}")
    return yaml_path


def backup_epoch(weight_name: str):
    def _hook(trainer):
        if not DRIVE_MOUNTED:
            return
        best = Path(trainer.save_dir) / "weights" / "best.pt"
        if not best.exists():
            return
        target = Path("/content/drive/MyDrive/cityguard")
        target.mkdir(parents=True, exist_ok=True)
        shutil.copy(best, target / weight_name)
        print(f"Drive: {weight_name} po epoce {trainer.epoch + 1}")

    return _hook


def fit_yolo(data_yaml: Path) -> Path:
    import torch

    def fit(current_batch: int):
        model = YOLO("yolo12n.pt")
        model.add_callback("on_fit_epoch_end", backup_epoch("best_litter.pt"))
        model.train(
            data=str(data_yaml),
            epochs=EPOCHS,
            imgsz=IMGSZ,
            batch=current_batch,
            device=0,
            project="/content/runs",
            name="litter",
            exist_ok=True,
            patience=10,
            workers=2,
            plots=True,
            save_period=5,
            cos_lr=True,
        )
        return model

    try:
        model = fit(BATCH)
    except RuntimeError as exc:
        if "out of memory" not in str(exc).lower():
            raise
        print("CUDA OOM. Powtarzam z batch=8.")
        torch.cuda.empty_cache()
        model = fit(8)

    best = Path("/content/runs/litter/weights/best.pt")
    out = Path("/content/best_litter.pt")
    shutil.copy(best, out)
    Path("/content/classes_litter.json").write_text(
        json.dumps(
            {"task": "litter", "conf": CONF, "names": {str(k): v for k, v in model.names.items()}},
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    print("Wagi:", out)
    return out


DRIVE_MOUNTED = False
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        DRIVE_MOUNTED = True
    except Exception as exc:
        print("Drive nie wszedł — trenuję bez kopii na Dysku. Wagi będą w /content.")
        print(exc)

images_zip = download_zenodo("images.zip")
ann_zip = download_zenodo("annotations.zip")
img_dir = unzip_once(images_zip, RAW / "images")
ann_dir = unzip_once(ann_zip, RAW / "annotations")
fit_yolo(link_split(coco_to_pairs(ann_dir, img_dir), DEST, CLASS_NAMES))

## Pobranie wag

Rozpakuj ZIP do `ml/weights/`. Inferencja szuka `best_litter.pt`.

In [ ]:
import zipfile
from pathlib import Path

from google.colab import files

bundle = Path("/content/cityguard_best_litter.pt.zip")
with zipfile.ZipFile(bundle, "w") as archive:
    found = False
    for name in ("best_litter.pt", "classes_litter.json"):
        path = Path("/content") / name
        if path.exists():
            archive.write(path, name)
            found = True
            print("pakuję", name)
        else:
            print("pomijam", name)
if not found:
    raise SystemExit("Nie ma jeszcze wag. Najpierw odpal trening.")
files.download(str(bundle))

## Podgląd na klatkach z demo

Ustaw `RUN_PREVIEW = True` i uruchom komórkę ręcznie. Próg: `CONF` (0.35).

In [ ]:
RUN_PREVIEW = False

if not RUN_PREVIEW:
    print("Podgląd wyłączony. Ustaw RUN_PREVIEW = True i odpal tę komórkę ponownie.")
else:
    from pathlib import Path

    from google.colab import files
    from ultralytics import YOLO

    preview = Path("/content/preview_images")
    preview.mkdir(exist_ok=True)
    for name, blob in files.upload().items():
        (preview / name).write_bytes(blob)
    YOLO("/content/best_litter.pt").predict(
        source=str(preview),
        conf=CONF,
        save=True,
        project="/content/preview_out",
        name="litter",
        exist_ok=True,
    )
    print("Obrazki z boxami: /content/preview_out/litter")